# Adversarial Attacks (FGSM, PGD)

Evaluates adversarial robustness of all four quantisation variants using the Adversarial Robustness Toolbox (ART).

**Two attack settings, both included:**
1. **White-box:** attacks crafted directly against each model's own gradients (FP32 baseline, QAT fake-quant model, Binary fake-quant model)
2. **Transfer:** adversarial examples crafted on the FP32 baseline are fed into each variant's `.tflite` model (the real PTQ/QAT/Binary artifacts), to measure how well an attack crafted without knowledge of quantisation transfers to the deployed model.

**Metrics:** adversarial accuracy and Attack Success Rate (ASR), swept across epsilon = [0.01, 0.03 (~8/255), 0.05, 0.1], for both FGSM and PGD.

ASR is computed over samples the model originally classified correctly

In [ ]:
!pip install adversarial-robustness-toolbox --quiet

In [ ]:
# import
import tensorflow as tf
import numpy as np
import os
import csv

from art.estimators.classification import TensorFlowV2Classifier
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/tinyml-quant-security'

In [ ]:
# Load CIFAR-10 test set
(_, _), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_test = x_test.astype('float32') / 255.0
y_test = y_test.flatten()

# Use a fixed subset for attack generation, PGD especially is expensive to run
# on the full 10,000-sample test set across 2 attacks x 4 epsilons x 3 models.
N_ATTACK_SAMPLES = 1000
rng = np.random.RandomState(SEED)
attack_idx = rng.choice(len(x_test), N_ATTACK_SAMPLES, replace=False)
x_attack = x_test[attack_idx]
y_attack = y_test[attack_idx]

print(f"Attack subset: {x_attack.shape}")

## Re-declare custom layers

In [ ]:
def make_fake_quantize(num_bits=8):
    qmax = float(2 ** (num_bits - 1) - 1)

    @tf.custom_gradient
    def fq(x):
        scale = tf.maximum(tf.reduce_max(tf.abs(x)), 1e-8) / qmax
        x_q = tf.round(x / scale) * scale
        def grad(dy, variables=None):
            if variables:
                return dy, [tf.zeros_like(v) for v in variables]
            return dy
        return x_q, grad
    return fq

fake_quantize_weights = make_fake_quantize(num_bits=8)
fake_quantize_activations = make_fake_quantize(num_bits=8)


@tf.custom_gradient
def binarize_ste(w):
    w_binarized = tf.sign(w)
    w_binarized = tf.where(tf.equal(w_binarized, 0), tf.ones_like(w_binarized), w_binarized)
    def grad(dy, variables=None):
        mask = tf.cast(tf.abs(w) <= 1.0, dy.dtype)
        dx = dy * mask
        if variables:
            return dx, [tf.zeros_like(v) for v in variables]
        return dx
    return w_binarized, grad


class QATConv2D(tf.keras.layers.Conv2D):
    def call(self, inputs):
        q_kernel = fake_quantize_weights(self.kernel)
        outputs = tf.nn.conv2d(inputs, q_kernel, strides=[1, *self.strides, 1], padding=self.padding.upper())
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs


class QATDense(tf.keras.layers.Dense):
    def call(self, inputs):
        q_kernel = fake_quantize_weights(self.kernel)
        outputs = tf.matmul(inputs, q_kernel)
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs


class FakeQuantActivation(tf.keras.layers.Layer):
    def call(self, inputs):
        return fake_quantize_activations(inputs)


class BinaryConv2D(tf.keras.layers.Conv2D):
    def call(self, inputs):
        q_kernel = binarize_ste(self.kernel)
        outputs = tf.nn.conv2d(inputs, q_kernel, strides=[1, *self.strides, 1], padding=self.padding.upper())
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs


class BinaryDense(tf.keras.layers.Dense):
    def call(self, inputs):
        q_kernel = binarize_ste(self.kernel)
        outputs = tf.matmul(inputs, q_kernel)
        if self.use_bias:
            outputs = tf.nn.bias_add(outputs, self.bias)
        if self.activation is not None:
            outputs = self.activation(outputs)
        return outputs


custom_objects = {
    'QATConv2D': QATConv2D,
    'QATDense': QATDense,
    'FakeQuantActivation': FakeQuantActivation,
    'BinaryConv2D': BinaryConv2D,
    'BinaryDense': BinaryDense,
}

In [ ]:
# Load the three differentiable models for white-box attacks
fp32_model = tf.keras.models.load_model(f'{PROJECT_DIR}/models/baseline.keras')
qat_float_model = tf.keras.models.load_model(f'{PROJECT_DIR}/models/qat_model.keras', custom_objects=custom_objects)
binary_float_model = tf.keras.models.load_model(f'{PROJECT_DIR}/models/binary_model.keras', custom_objects=custom_objects)

print("Loaded FP32, QAT (fake-quant float), and Binary (fake-quant float) models for white-box attacks.")

In [ ]:
# Wrap each model as an ART classifier
loss_object = tf.keras.losses.SparseCategoricalCrossentropy()

def make_art_classifier(keras_model):
    return TensorFlowV2Classifier(
        model=keras_model,
        nb_classes=10,
        input_shape=(32, 32, 3),
        loss_object=loss_object,
        clip_values=(0.0, 1.0),
    )

art_classifiers = {
    'FP32': make_art_classifier(fp32_model),
    'QAT': make_art_classifier(qat_float_model),
    'Binary': make_art_classifier(binary_float_model),
}

## White-box attack

For each model, for each attack (FGSM, PGD), for each epsilon: craft adversarial examples using that model's own gradients, then evaluate adversarial accuracy and ASR against that same model.

In [ ]:
EPSILON_VALUES = [0.01, 8/255, 0.05, 0.1]  # 8/255 ~ 0.0314, the Madry et al. convention
PGD_ITERATIONS = 20

def compute_accuracy_and_asr(classifier, x_clean, x_adv, y_true):
    clean_preds = np.argmax(classifier.predict(x_clean), axis=1)
    adv_preds = np.argmax(classifier.predict(x_adv), axis=1)

    correctly_classified_mask = (clean_preds == y_true)
    n_correct_clean = np.sum(correctly_classified_mask)

    adv_accuracy = np.mean(adv_preds == y_true)

    if n_correct_clean > 0:
        flipped = np.sum((adv_preds != y_true) & correctly_classified_mask)
        asr = flipped / n_correct_clean
    else:
        asr = float('nan')

    return adv_accuracy, asr

In [ ]:
whitebox_results = []

for model_name, classifier in art_classifiers.items():
    clean_preds = np.argmax(classifier.predict(x_attack), axis=1)
    clean_acc = np.mean(clean_preds == y_attack)
    print(f"\n=== {model_name} - clean accuracy on attack subset: {clean_acc:.4f} ===")

    for eps in EPSILON_VALUES:
        # FGSM
        fgsm = FastGradientMethod(estimator=classifier, eps=eps)
        x_adv_fgsm = fgsm.generate(x=x_attack)
        fgsm_acc, fgsm_asr = compute_accuracy_and_asr(classifier, x_attack, x_adv_fgsm, y_attack)

        whitebox_results.append({
            'model': model_name, 'attack': 'FGSM', 'epsilon': eps,
            'adv_accuracy': fgsm_acc, 'asr': fgsm_asr, 'mode': 'white-box'
        })
        print(f"  FGSM eps={eps:.4f} - adv_acc={fgsm_acc:.4f}, ASR={fgsm_asr:.4f}")

        # PGD
        pgd = ProjectedGradientDescent(
            estimator=classifier, eps=eps, eps_step=eps / 4,
            max_iter=PGD_ITERATIONS, num_random_init=1
        )
        x_adv_pgd = pgd.generate(x=x_attack)
        pgd_acc, pgd_asr = compute_accuracy_and_asr(classifier, x_attack, x_adv_pgd, y_attack)

        whitebox_results.append({
            'model': model_name, 'attack': 'PGD', 'epsilon': eps,
            'adv_accuracy': pgd_acc, 'asr': pgd_asr, 'mode': 'white-box'
        })
        print(f"  PGD  eps={eps:.4f} - adv_acc={pgd_acc:.4f}, ASR={pgd_asr:.4f}")

## Transfer attack: FP32-crafted adversarial examples vs. deployed TFLite models

Tests how well attacks crafted without knowledge of quantisation transfer to the actual deployed PTQ/QAT/Binary `.tflite`. This is the more realistic threat model for a deployed TinyML device, where an attacker likely doesn't have white-box access to the exact quantised weights running on-device.

In [ ]:
def quantize_input_int8(x_float, scale, zero_point):
    x_int8 = x_float / scale + zero_point
    return np.clip(np.round(x_int8), -128, 127).astype(np.int8)


def predict_tflite(model_path, x_batch, is_int8):
    interpreter = tf.lite.Interpreter(model_path=model_path)
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    preds = []
    if is_int8:
        scale, zero_point = input_details['quantization']

    for i in range(len(x_batch)):
        sample = x_batch[i:i+1].astype(np.float32)
        if is_int8:
            sample = quantize_input_int8(sample, scale, zero_point)
        interpreter.set_tensor(input_details['index'], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details['index'])
        preds.append(np.argmax(output[0]))

    return np.array(preds)

In [ ]:
deployed_models = {
    'PTQ': {'path': f'{PROJECT_DIR}/models/ptq_model.tflite', 'is_int8': True},
    'QAT': {'path': f'{PROJECT_DIR}/models/qat_model.tflite', 'is_int8': True},
    'Binary': {'path': f'{PROJECT_DIR}/models/binary_model.tflite', 'is_int8': False},
}

transfer_results = []
fp32_classifier = art_classifiers['FP32']

for eps in EPSILON_VALUES:
    fgsm = FastGradientMethod(estimator=fp32_classifier, eps=eps)
    x_adv_fgsm = fgsm.generate(x=x_attack)

    pgd = ProjectedGradientDescent(
        estimator=fp32_classifier, eps=eps, eps_step=eps / 4,
        max_iter=PGD_ITERATIONS, num_random_init=1
    )
    x_adv_pgd = pgd.generate(x=x_attack)

    for deployed_name, cfg in deployed_models.items():
        clean_preds_deployed = predict_tflite(cfg['path'], x_attack, cfg['is_int8'])
        correctly_classified_mask = (clean_preds_deployed == y_attack)
        n_correct_clean = np.sum(correctly_classified_mask)

        for attack_name, x_adv in [('FGSM', x_adv_fgsm), ('PGD', x_adv_pgd)]:
            adv_preds_deployed = predict_tflite(cfg['path'], x_adv, cfg['is_int8'])
            adv_acc = np.mean(adv_preds_deployed == y_attack)

            if n_correct_clean > 0:
                flipped = np.sum((adv_preds_deployed != y_attack) & correctly_classified_mask)
                asr = flipped / n_correct_clean
            else:
                asr = float('nan')

            transfer_results.append({
                'model': deployed_name, 'attack': attack_name, 'epsilon': eps,
                'adv_accuracy': adv_acc, 'asr': asr, 'mode': 'transfer-from-FP32'
            })
            print(f"Transfer -> {deployed_name} | {attack_name} eps={eps:.4f} - adv_acc={adv_acc:.4f}, ASR={asr:.4f}")

In [ ]:
# Save all results (white-box + transfer) to a CSV
all_results = whitebox_results + transfer_results
results_path = f'{PROJECT_DIR}/results/adversarial_eval.csv'

with open(results_path, 'w', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=['model', 'attack', 'epsilon', 'adv_accuracy', 'asr', 'mode'])
    writer.writeheader()
    for row in all_results:
        writer.writerow(row)

print(f"Saved {len(all_results)} rows to {results_path}")